# 07 — Demographics vs. Homeless Rate

Examines correlations between state-level homeless rates and:
- **Race/ethnicity** (% Black — Census ACS 2022)
- **Veteran population** (% veterans — Census ACS 2022)
- **Veteran over-representation** — are veterans disproportionately homeless relative to their population share?


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_state_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'pct_black', 'veteran_pct'])
print(f'Loaded {len(df)} states')


Loaded 51 states


In [2]:
slope, intercept, r, p, se = stats.linregress(df['pct_black'], df['homeless_rate_per_10k'])
r2 = r ** 2
print(f'% Black vs homeless rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df['pct_black'].min(), df['pct_black'].max()]
y_range = [slope * x + intercept for x in x_range]

fig1 = px.scatter(
    df, x='pct_black', y='homeless_rate_per_10k', text='state_postal',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title=f'% Black Population vs. Homeless Rate by State<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} (Census ACS 2022)</sup>',
    labels={'pct_black': '% Black (non-Hispanic)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
)
fig1.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.show()


% Black vs homeless rate: r=0.105, R²=0.011, p=0.4654


In [3]:
slope2, intercept2, r2v, p2, se2 = stats.linregress(df['veteran_pct'], df['homeless_rate_per_10k'])
r2_2 = r2v ** 2
print(f'Veteran % vs homeless rate: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')

x_range2 = [df['veteran_pct'].min(), df['veteran_pct'].max()]
y_range2 = [slope2 * x + intercept2 for x in x_range2]

fig2 = px.scatter(
    df, x='veteran_pct', y='homeless_rate_per_10k', text='state_postal',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title=f'Veteran Population % vs. Homeless Rate by State<br><sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f} (Census ACS 2022)</sup>',
    labels={'veteran_pct': 'Veteran Population (%)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
)
fig2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
fig2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig2.show()


Veteran % vs homeless rate: r=-0.413, R²=0.171, p=0.0026


In [4]:
# Veteran over-representation: (veteran % of homeless) / (veteran % of population)
# Ratio > 1 = veterans are over-represented in homelessness
df2 = df.dropna(subset=['veteran_homeless', 'total_homeless', 'veteran_pct'])
df2 = df2.copy()
df2['veteran_share_of_homeless_pct'] = (df2['veteran_homeless'] / df2['total_homeless'] * 100).round(2)
df2['overrep_ratio'] = (df2['veteran_share_of_homeless_pct'] / df2['veteran_pct']).round(2)

print(f'National avg veteran share of homeless: {df2["veteran_share_of_homeless_pct"].mean():.1f}%')
print(f'National avg veteran share of population: {df2["veteran_pct"].mean():.1f}%')
print(f'National avg over-representation ratio: {df2["overrep_ratio"].mean():.2f}x')

fig3 = px.bar(
    df2.sort_values('overrep_ratio', ascending=True),
    x='overrep_ratio', y='state_name', orientation='h',
    color='overrep_ratio',
    color_continuous_scale='RdYlGn_r',
    title='Veteran Over-Representation in Homelessness by State<br>' +
          '<sup>(veteran % of homeless) ÷ (veteran % of population) — ratio > 1 means veterans are disproportionately homeless</sup>',
    labels={'overrep_ratio': 'Over-representation ratio', 'state_name': ''},
)
fig3.add_vline(x=1.0, line_dash='dash', line_color='black', annotation_text='Proportional')
fig3.update_layout(height=900)
fig3.show()


National avg veteran share of homeless: 9.7%
National avg veteran share of population: 7.0%
National avg over-representation ratio: 1.39x


### Political Ideology vs. Homeless Rate

2024 presidential election results (Harris 2-party %) used as a proxy for state political ideology.
Margin = Dem% × 2 − 100: positive = Democratic lean, negative = Republican.


In [5]:
df_pol = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_state_data.csv')
df_pol = df_pol.dropna(subset=['homeless_rate_per_10k', 'dem_pres_margin_2024'])

slope, intercept, r, p, se = stats.linregress(df_pol['dem_pres_margin_2024'], df_pol['homeless_rate_per_10k'])
r2 = r ** 2
print(f'2024 Dem margin vs homeless rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df_pol['dem_pres_margin_2024'].min(), df_pol['dem_pres_margin_2024'].max()]
y_range = [slope * x + intercept for x in x_range]

fig_pol = px.scatter(
    df_pol, x='dem_pres_margin_2024', y='homeless_rate_per_10k', text='state_postal',
    color='dem_pres_margin_2024',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title=f'2024 Presidential Vote (Dem margin) vs. Homeless Rate by State<br>'
          f'<sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} — more Democratic states tend to have higher homeless rates</sup>',
    labels={'dem_pres_margin_2024': 'Dem margin (2024 presidential, % points)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
    hover_data={'dem_pres_margin_2024': ':.1f'},
)
fig_pol.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression',
    line=dict(color='black', dash='dash')))
fig_pol.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig_pol.add_vline(x=0, line_dash='dot', line_color='grey', annotation_text='Even split')
fig_pol.show()


2024 Dem margin vs homeless rate: r=0.681, R²=0.464, p=0.0000


In [6]:
# Political lean vs. UNSHELTERED % (holding policy constant)
slope2, intercept2, r2v, p2, se2 = stats.linregress(df_pol['dem_pres_margin_2024'], df_pol['unsheltered_pct'])
r2_2 = r2v ** 2
print(f'2024 Dem margin vs unsheltered %: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')

x_range2 = [df_pol['dem_pres_margin_2024'].min(), df_pol['dem_pres_margin_2024'].max()]
y_range2 = [slope2 * x + intercept2 for x in x_range2]

fig_pol2 = px.scatter(
    df_pol, x='dem_pres_margin_2024', y='unsheltered_pct', text='state_postal',
    color='dem_pres_margin_2024',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title=f'2024 Presidential Vote (Dem margin) vs. % Unsheltered Homeless by State<br>'
          f'<sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}</sup>',
    labels={'dem_pres_margin_2024': 'Dem margin (2024 presidential, % points)', 'unsheltered_pct': '% of Homeless Who Are Unsheltered'},
    hover_name='state_name',
)
fig_pol2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression',
    line=dict(color='black', dash='dash')))
fig_pol2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig_pol2.add_vline(x=0, line_dash='dot', line_color='grey')
fig_pol2.show()


2024 Dem margin vs unsheltered %: r=-0.206, R²=0.042, p=0.1468
